### Plan agreed with Claude -- to review before writing any code

#### What the question actually asks for

"At what time-interval would you be least certain about how many runs there will be in
the game? (Choose a 1-minute time interval -- there can be many answers here)."

So the deliverable is **a specific one-minute window**, quoted as a timestamp, not a
method. The "many answers" hint tells us different reasonable measures will select
different minutes, and that showing that disagreement -- with the reasoning for each --
is the point rather than a weakness. We will therefore report the winning minute under
each measure separately, and then nominate one as the headline answer.

#### Universe: the KXMLBTOTAL chain, 11 strikes

`KXMLBTOTAL` is runs scored in the game by both teams combined, which is exactly what
the question asks about. `F5TOTAL` covers only five innings, `TEAMTOTAL` covers one
team, and `RFI` covers one inning, so none of them answer it. Strikes run n = 2 to 12,
giving 11 markets.

#### Step 1 -- survival curve to PMF

Each contract is a survival probability S(n) = P(runs >= n). Differencing adjacent
strikes gives the probability mass function. With strikes only spanning 2 to 12, the
result has 12 buckets, two of which are censored lumps:

| bucket | formula | size observed in this dataset |
| --- | --- | --- |
| runs <= 1 | 1 - S(2) | 3.0% - 3.5% |
| runs = n, for n = 2..11 | S(n) - S(n+1) | the interior |
| runs >= 12 | S(12) | 14.5% - 15.5% |

The head lump is small enough to ignore. The tail lump is not -- roughly 15% of the
mass sits in an open-ended bucket, and we cannot know how it splits across 12, 13, 14+.
This is the single biggest reason the choice of metric below matters.

Q1 established there are no monotonicity violations anywhere in this chain, so every
differenced bucket is guaranteed non-negative and the PMF is always well formed. We do
not need to clip or renormalise.

#### Step 2 -- the three metrics, and why these

**(1) Entropy of the PMF -- how wide is the market's belief?**  `H = -sum p*log2(p)`,
in bits.

This is the information-theoretic definition of uncertainty, which is what the question
is asking about. Its decisive advantage here is that entropy is invariant to what the
outcomes are *called* -- it only cares how mass is spread across buckets. So the
`runs >= 12` lump is simply one bucket among twelve and **never has to be assigned a
numeric value**. That removes the tail problem entirely rather than papering over it.

The cost of that same property: entropy does not know that 5 runs is nearer to 6 than to
12. A distribution on {5,6} and one on {2,12} have identical entropy but very different
uncertainty about *how many* runs. That is the one thing variance knows and entropy does
not, which is why variance is kept below as a sensitivity check rather than deleted.

**(2) Volatility of E[runs] -- how hard is the market revising?**  Within each minute,
the standard deviation (and the high-low range) of E[runs] = sum[n*p(n)]

Entropy measures how wide the market's belief is; this measures how unstable it is. If
E[runs] sits at 7.3 for an hour and then thrashes between 7.2 and 7.5 inside one
minute, that minute is one where the market is actively disagreeing with itself.
Because the whole dataset is pregame -- no runs have been scored, so nothing can update
the distribution except lineups, weather and order flow -- we expect the *level*
measures to be nearly flat and the *churn* measure to carry most of the discrimination.
Diagnostics below confirm this.

E[runs] does require a numeric value for the `>= 12` bucket. We will place it at 13 and
report the sensitivity of the answer to 12 / 13 / 14, the same way Q1 reported
sensitivity to the freshness threshold.

**(3) Mean staleness of the PMF -- how well do we even observe the market?**  Walking
forward in time, keep 11 counters, one per strike, each holding the age of that strike's
most recent quote. The PMF's staleness at any instant is the **average** of those 11.

This measures something genuinely different from (1) and (2): not the market's
uncertainty about runs, but *our* uncertainty about where the market is. Both are
legitimate readings of "least certain", and the question inviting many answers is
licence to include it. A minute in which no strike has been quoted for several minutes
is a minute in which we could not confidently price anything, which is a real and
practical form of uncertainty.

*Why average and not max.* Consider two edge cases. In 1-minute interval A, we saw 10 of
the 11 strikes quoted just 1 minute ago, and the 11th strike is a laggard where we saw
its quote 10 minutes ago. Max would call this whole PMF 10 minutes stale -- but in
reality, we can be pretty sure where the 11th strike should be priced, given information
about the other 10 strikes (some interpolation based on how historically those 11
strikes relate to each other should solve for the 11th missing strike). Now consider
1-minute interval B, where all 11 strikes are missing and all of them were last seen 7
minutes ago. Max would rank this PMF as being 7 minutes stale, which is fresher and more
up to date than A. But I have seen zero strikes quoted for the past 7 minutes, and I
could argue convincingly well that I know less about where the market is in case B.
Average gets both right: A scores 1.8 minutes and B scores 7.0.

Q1 supports the interpolation argument in case A directly: the chain was internally
consistent, correctly nested, with no arbitrage anywhere, and even the never-traded PIT
strikes were perfectly coherent. That is the signature of one maker pricing the whole
ladder off a single fitted distribution, which is exactly the condition under which 10
observed strikes pin down the 11th.

A mass-weighted version -- weighting each strike by the probability mass it governs, so
a stale S(7) counts for more than a stale S(12) -- is a sensible refinement to add
later. Start with the simple average.

**Sensitivity check, not a headline metric: variance of the PMF.**
`Var = sum n^2*p(n) - (sum n*p(n))^2`. Reported alongside entropy to show the two agree,
but not used as the primary answer, because variance weights by squared distance from
the mean and roughly 15% of the mass is in an open-ended tail bucket. The answer would
partly reflect our invented tail placement rather than the market.

#### A measure considered and dropped: bid-ask spread width

We considered bid-ask spread width as a possible candidate -- makers widen their quotes
when they are uncertain -- but given that **96.4% of book rows on this chain are 1 cent
wide and the remaining 3.6% are 2 cents**, with nothing wider, there is no variation to
exploit. A measure taking two distinct values across 21,543 observations cannot
meaningfully rank 360 minutes, so it is not used.

#### Step 3 -- when is the PMF even defined?

Entropy, E[runs] and variance all require **all 11 strikes simultaneously** -- a ladder
cannot be differenced if it is only partly observed. But the strikes do not update
together: they are quoted at different times, so at most instants some are current and
others are minutes old. Evaluating on a fixed clock grid would silently forward-fill
arbitrarily old quotes into the calculation, which is the thing Q1 established we should
not do.

Instead, the PMF is **only defined when every one of the 11 strikes is fresher than a
staleness threshold**, set to **10 seconds** by default and exposed as a parameter so
the whole analysis can be re-run at other values. Same discipline as the Q1 arbitrage
check: only compare quotes we actually observed close together in time.

Coverage at various thresholds, measured on this dataset:

| threshold | valid instants | share of chain events | minutes with >= 1 valid observation |
| --- | --- | --- | --- |
| <= 1s | 3,573 | 9.5% | 202 / 360 (56.1%) |
| <= 2s | 4,415 | 11.8% | 207 / 360 (57.5%) |
| <= 5s | 6,516 | 17.4% | 223 / 360 (61.9%) |
| **<= 10s** | **9,961** | **26.5%** | **259 / 360 (71.9%)** |
| <= 30s | 24,452 | 65.1% | 321 / 360 (89.2%) |
| <= 60s | 31,715 | 84.5% | 339 / 360 (94.2%) |

At the 10-second default, **101 of the 360 minutes have no valid PMF at all**, so
entropy and E[runs] volatility are undefined there and are reported as missing rather
than filled.

That is not a defect, and it is worth stating in the answer: the minutes where the PMF
is undefined are exactly the stale ones, which are the leading candidates to win on
metric (3). The three metrics therefore partition the window rather than compete over
it -- (1) and (2) rank the minutes where we could see the market clearly, and (3) ranks
the minutes where we could not.

#### Step 4 -- walk-forward logic over 1-minute intervals

1. Take `start_time` and `end_time` as the min and max of `recv_ts_utc` across the
   `KXMLBTOTAL` chain. Here that is 17:35:34 to 23:34:41 UTC, 5.99 hours, giving
   **360 one-minute bins**. Use `recv_ts_utc`, not exchange time, because that is when
   the information was actually available to us.
2. Build left-closed, right-open bins: `[start, start+1min)`, `[start+1min,
   start+2min)`, ... through `end_time`. Label each bin by its left edge.
3. Walk the book feed once in chronological order, holding a dict of the latest quote
   per strike plus its last-seen timestamp, exactly as the Q1 arbitrage engine does.
   Evaluate on each arriving chain event rather than on a fixed clock grid -- the feed
   is event-driven, so this samples the market when it actually moved. The dict holds
   the last quote seen per strike and its timestamp; it is a lookup of what we last
   observed, not a resampled or filled time series.
4. At each event, compute the age of all 11 strikes. Record the mean age
   unconditionally -- that is metric (3), and it is defined at every instant after
   warm-up. If and only if the **maximum** age is within the freshness threshold,
   additionally compute the PMF and from it entropy, E[runs] and variance.
5. Aggregate to the minute: entropy = mean of valid observations in the bin; E[runs]
   volatility = standard deviation and high-low range of valid observations in the bin;
   staleness = mean across all observations in the bin. Bins with no valid PMF
   observation get NaN for the first two and a real value for the third.
6. Rank the bins under each metric, report the top few for each, and note where they
   agree and disagree.

#### Assumptions, stated up front

- **Mid prices.** Price at any instant is `(best_bid + best_ask) / 2`. The chain is
  1 cent wide 96.4% of the time so the mid is a fair summary, and Q1 already established
  that mids are the right screen for chain-level structure.
- **What "using a price" means here.** There is no forward fill in the pandas sense --
  nothing is resampled onto a timestamp index and carried forward. We hold a dict with
  one entry per strike containing the last quote we actually saw for it and when we saw
  it. When all 11 strikes pass the staleness test -- meaning we last saw a quote for
  every one of them within the past 10 seconds -- we recognise those 11 probabilities as
  fresh enough to use, and compute E[runs], entropy and variance from them. If even one
  strike fails the test, no PMF is computed at that instant.

  This matters because the feed does not report every change: Q1 found that 66% of
  trades land inside book-feed gaps of more than 5 seconds. The staleness test is what
  bounds our exposure to changes we did not see, and metric (3) reports that exposure
  directly rather than hiding it.
- **Warm-up.** The first bin cannot be evaluated until all 11 strikes have been quoted
  at least once. Bins before that are dropped, not zero-filled.
- **Quiet bins.** A minute containing no quotes at all on the chain is still evaluated
  for metric (3), using the running ages -- that is precisely the case metric (3) exists
  to flag, so dropping such bins would discard the signal.
- **Tail placement.** Only metrics that need a number for the `>= 12` bucket (E[runs],
  variance) are affected. Default 13, with 12 and 14 reported as sensitivity. Entropy
  needs no such assumption.
- **Pregame only.** The window ends at 23:34 UTC and first pitch is 23:40 UTC, so no
  runs are scored anywhere in this dataset. Whatever minute we choose, it is a minute of
  pregame uncertainty, and the write-up should say so rather than imply we observed
  in-game repricing.

#### What we already know from diagnostics

Computed over the full window before committing to this plan:

| measure | min | max | total swing |
| --- | --- | --- | --- |
| entropy | 3.4048 bits | 3.4657 bits | 1.8% |
| E[runs] | 7.3000 | 7.4350 | 1.8% (a range of 0.135 runs) |
| variance | 11.4860 | 11.9100 | 3.6% |

The market's belief is close to frozen across six hours, which is what a pregame window
with no observable events should look like. This has a direct consequence for how the
answer is phrased: the highest-entropy minute wins by a hair, not by a mile, so the
honest claim is "belief width was essentially constant and here is where it was widest",
not "the market was much less certain at this minute". The churn and staleness measures
are expected to discriminate more sharply, and the worst observed gap on any single
strike is 6.8 minutes, which bounds how extreme metric (3) can get.